# YouTube Video Statistics Analysis Using Hadoop and Apache Spark
**An End-to-End Big Data Engineering, Spark SQL, MLlib & Analytics Pipeline**

---
### Pipeline Workflow Overview:
1. **Environment Setup & SparkSession Initialization**
2. **Data Ingestion**: Multi-country CSVs with multiline and quote escaping
3. **Data Cleansing**: Deduplication, null filtering, type casting, date parsing (`yy.dd.MM` & ISO 8601)
4. **Category Mapping**: JSON hierarchy extraction and broadcast join
5. **Partitioned Parquet Storage**: Columnar storage with Snappy compression
6. **Feature Engineering**: Like ratios, engagement rate, days to trend, temporal & text metrics
7. **Spark SQL Analysis**: 8 multidimensional queries (Top videos, categories, countries, timing, correlations)
8. **Spark MLlib Machine Learning**: Random Forest view prediction and K-Means engagement clustering
9. **Visualizations**: High-resolution Matplotlib and Seaborn plots
10. **Performance Tuning**: CSV vs Parquet benchmarks and in-memory caching speedup


## 1. SparkSession Initialization and Environment Configuration

In [ ]:
import os
import sys
import time
import json
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from config import config  # Automatically configures Java 17, Hadoop winutils, and worker paths
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import LongType, IntegerType, BooleanType, DoubleType
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Initialize SparkSession with optimized configs
spark = (SparkSession.builder
    .appName("YouTube_Statistics_Jupyter_Analysis")
    .master("local[*]")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print(f"SparkSession Active! Version: {spark.version}")

## 2. Ingestion of Multi-Country Raw Datasets
Reading raw CSV files with `multiLine=True` to prevent record splits on multiline descriptions, extracting the `country` identifier from file paths.

In [ ]:
raw_dir = PROJECT_ROOT / "data" / "raw"
csv_pattern = str(raw_dir / "*videos.csv")

raw_df = (spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(csv_pattern, inferSchema=True))

# Extract country code dynamically from file path
df = raw_df.withColumn(
    "country",
    F.upper(F.coalesce(
        F.regexp_extract(F.input_file_name(), r"([A-Za-z]{2})(?:videos|_api)", 1),
        F.lit("US")
    ))
)

print(f"Total Raw Records Ingested: {df.count():,}")
df.select("country", "video_id", "title", "channel_title", "views").show(5, truncate=30)

## 3. Data Cleansing, Type Casting & Date Parsing
- Deduplicating snapshots on `(video_id, trending_date)`
- Parsing `trending_date` in Kaggle format (`yy.dd.MM` e.g., `17.14.11` -> `2017-11-14`)
- Parsing ISO 8601 `publish_time` timestamp

In [ ]:
# Deduplicate on video_id and trending_date
df = df.dropDuplicates(["video_id", "trending_date"])
df = df.filter(F.col("video_id").isNotNull() & (F.length(F.trim(F.col("video_id"))) > 0))

# Cast numeric columns
for col_name in ["views", "likes", "dislikes", "comment_count", "category_id"]:
    df = df.withColumn(col_name, F.col(col_name).cast(LongType()))

# Cast boolean flags
for bool_col in ["comments_disabled", "ratings_disabled", "video_error_or_removed"]:
    if bool_col in df.columns:
        df = df.withColumn(bool_col, F.col(bool_col).cast(BooleanType()))

# Date Parsing
df = df.withColumn("trending_date_parsed", F.coalesce(
    F.to_date(F.col("trending_date"), "yy.dd.MM"),
    F.to_date(F.col("trending_date"), "yyyy-MM-dd")
))

df = df.withColumn("publish_timestamp", F.coalesce(
    F.to_timestamp(F.col("publish_time"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"),
    F.to_timestamp(F.col("publish_time"))
))
df = df.withColumn("publish_date", F.to_date(F.col("publish_timestamp")))

print(f"Records after Deduplication and Cleansing: {df.count():,}")

## 4. Category Mapping via JSON Broadcast Join

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType

# Load JSON category files
cat_files = list(raw_dir.glob("*category_id.json"))
cat_mappings = []
for cf in cat_files:
    c_code = cf.name.split("_")[0].upper()
    with open(cf, "r", encoding="utf-8") as f:
        data = json.load(f)
        for it in data.get("items", []):
            cat_mappings.append((c_code, int(it["id"]), it["snippet"]["title"]))

schema = StructType([
    StructField("cat_country", StringType(), False),
    StructField("cat_id", LongType(), False),
    StructField("category_title", StringType(), False)
])
cat_df = spark.createDataFrame(cat_mappings, schema).distinct()

# Join category title
df = df.join(
    cat_df,
    (df.country == cat_df.cat_country) & (df.category_id == cat_df.cat_id),
    "left"
).drop("cat_country", "cat_id")

df = df.fillna({"category_title": "General / Other"})
df.select("country", "title", "category_title").show(5, truncate=35)

## 5. Feature Engineering
Deriving interaction ratios, velocity, temporal indicators, and metadata metrics.

In [ ]:
safe_views = F.when(F.col("views") > 0, F.col("views")).otherwise(F.lit(1.0))

df = df.withColumn("like_ratio", F.round(F.col("likes") / safe_views, 6))
df = df.withColumn("dislike_ratio", F.round(F.col("dislikes") / safe_views, 6))
df = df.withColumn("comment_ratio", F.round(F.col("comment_count") / safe_views, 6))
df = df.withColumn("engagement_rate", F.round((F.col("likes") + F.col("dislikes") + F.col("comment_count")) / safe_views, 6))

# Velocity: Days to Trend
df = df.withColumn("days_to_trend", F.greatest(F.lit(0), F.datediff(F.col("trending_date_parsed"), F.col("publish_date"))))

# Temporal attributes
df = df.withColumn("publish_hour", F.hour(F.col("publish_timestamp")))
df = df.withColumn("publish_weekday", F.date_format(F.col("publish_timestamp"), "E"))
df = df.withColumn("publish_year_month", F.date_format(F.col("publish_timestamp"), "yyyy-MM"))

# Text metrics
df = df.withColumn("title_length", F.length(F.coalesce(F.col("title"), F.lit(""))))
df = df.withColumn("tag_count", F.when(
    (F.col("tags") == "[none]") | F.col("tags").isNull(), F.lit(0)
).otherwise(F.size(F.split(F.col("tags"), r"\|"))))

df.createOrReplaceTempView("youtube_features")
print("[+] Feature Engineering Complete!")
df.select("title", "like_ratio", "engagement_rate", "days_to_trend", "publish_weekday").show(5, truncate=30)

## 6. Spark SQL Analytical Queries

In [ ]:
# Top 10 Most Viewed Videos
top_10 = spark.sql("""
    SELECT country, title, channel_title, category_title, MAX(views) as max_views, MAX(likes) as max_likes
    FROM youtube_features
    GROUP BY country, title, channel_title, category_title
    ORDER BY max_views DESC
    LIMIT 10
""")
top_10.show(10, truncate=35)

# Category Aggregates
cat_summary = spark.sql("""
    SELECT category_title, COUNT(video_id) as video_count,
           ROUND(AVG(views), 0) as avg_views,
           ROUND(AVG(engagement_rate), 4) as avg_engagement_rate
    FROM youtube_features
    GROUP BY category_title
    ORDER BY avg_views DESC
""")
cat_summary.show(10, truncate=30)

## 7. Machine Learning with Spark MLlib (Random Forest & K-Means)

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import RegressionEvaluator, ClusteringEvaluator

# Feature indexing
cat_idx = StringIndexer(inputCol="category_title", outputCol="cat_idx", handleInvalid="keep").fit(df).transform(df)
country_idx = StringIndexer(inputCol="country", outputCol="country_idx", handleInvalid="keep").fit(cat_idx).transform(cat_idx)

feature_cols = ["likes", "comment_count", "days_to_trend", "title_length", "tag_count", "cat_idx", "country_idx"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
ml_data = assembler.transform(country_idx)

train, test = ml_data.randomSplit([0.8, 0.2], seed=42)
rf = RandomForestRegressor(featuresCol="features", labelCol="views", numTrees=30, seed=42)
rf_model = rf.fit(train)
preds = rf_model.transform(test)

r2_eval = RegressionEvaluator(labelCol="views", predictionCol="prediction", metricName="r2")
rmse_eval = RegressionEvaluator(labelCol="views", predictionCol="prediction", metricName="rmse")

print(f"Random Forest R2 Score: {r2_eval.evaluate(preds):.4f}")
print(f"Random Forest RMSE: {rmse_eval.evaluate(preds):,.2f}")

## 8. Visualizing Results with Matplotlib and Seaborn

In [ ]:
cat_pdf = cat_summary.toPandas().head(8)

fig, ax1 = plt.subplots(figsize=(10, 5))
sns.barplot(data=cat_pdf, x="category_title", y="avg_views", ax=ax1, palette="Blues_r")
ax1.set_title("Average Views by Top Categories")
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=30, ha="right")
ax1.set_ylabel("Average Views")
plt.tight_layout()
plt.show()

## 9. Pipeline Conclusion
This concludes the end-to-end big data processing, analytics, and machine learning pipeline for YouTube Trending Video Statistics using Apache Spark and Hadoop.